# Music Creation using a Multi-Agent AI System

**Copyright (c) 2026 Shrikara Kaudambady. All rights reserved.**

This notebook simulates a band of AI agents collaborating to compose a piece of music. We have an `OrchestratorAgent` that directs the band, and several `MusicianAgent`s (Drummer, Bassist, Melody) that generate their own parts based on a set of rules and probabilistic models. The final output is a MIDI file named `ai_composition.mid`.

### 1. Setup and Library Imports

In [ ]:
import mido
import numpy as np
import random

### 2. Music Theory and Composition Setup
We define the musical framework for our composition. This includes the key, scale, chord progression, and tempo. Modifying these variables will create a completely different song.

In [ ]:
KEY = 'C'
SCALE = ['C', 'D', 'E', 'F', 'G', 'A', 'B'] # C Major Scale
CHORD_PROGRESSION = ['C', 'G', 'Am', 'F'] # I-V-vi-IV in C Major
BPM = 120
TICKS_PER_BEAT = 480 # Standard for MIDI

# MIDI Note Numbers for our Key
note_map = {'C': 60, 'D': 62, 'E': 64, 'F': 65, 'G': 67, 'A': 69, 'B': 71}
scale_notes = [note_map[n] for n in SCALE]

# Define Chords (root, third, fifth)
chords = {
    'C': [note_map['C'], note_map['E'], note_map['G']],
    'G': [note_map['G'], note_map['B'], note_map['D']+12],
    'Am': [note_map['A'], note_map['C']+12, note_map['E']+12],
    'F': [note_map['F'], note_map['A'], note_map['C']+12]
}

### 3. Define the Musician Agents
Each agent has a `generate_notes` method that returns a list of MIDI messages for one measure.

In [ ]:
class DrummerAgent:
    """Generates a simple rock beat."""
    def generate_notes(self, measure, chord):
        notes = []
        # Kick on 1 and 3, Snare on 2 and 4
        notes.append(mido.Message('note_on', note=36, velocity=100, time=0, channel=9))
        notes.append(mido.Message('note_off', note=36, velocity=0, time=TICKS_PER_BEAT, channel=9))
        notes.append(mido.Message('note_on', note=38, velocity=90, time=0, channel=9))
        notes.append(mido.Message('note_off', note=38, velocity=0, time=TICKS_PER_BEAT, channel=9))
        notes.append(mido.Message('note_on', note=36, velocity=100, time=0, channel=9))
        notes.append(mido.Message('note_off', note=36, velocity=0, time=TICKS_PER_BEAT, channel=9))
        notes.append(mido.Message('note_on', note=38, velocity=90, time=0, channel=9))
        notes.append(mido.Message('note_off', note=38, velocity=0, time=TICKS_PER_BEAT, channel=9))
        
        # Hi-hats on 8th notes
        for i in range(8):
            notes.append(mido.Message('note_on', note=42, velocity=80, time=0 if i==0 else TICKS_PER_BEAT // 2, channel=9))
            notes.append(mido.Message('note_off', note=42, velocity=0, time=0, channel=9))
        return notes

class BassistAgent:
    """Generates a simple bassline following the root note of the chord."""
    def generate_notes(self, measure, chord):
        notes = []
        root_note = chords[chord][0] - 12 # Play one octave lower
        # Play root note on beat 1 and 3
        notes.append(mido.Message('note_on', note=root_note, velocity=80, time=0, channel=1))
        notes.append(mido.Message('note_off', note=root_note, velocity=0, time=TICKS_PER_BEAT * 2, channel=1))
        notes.append(mido.Message('note_on', note=root_note, velocity=80, time=0, channel=1))
        notes.append(mido.Message('note_off', note=root_note, velocity=0, time=TICKS_PER_BEAT * 2, channel=1))
        return notes

class MelodyAgent:
    """Generates a melodic line using a probabilistic random walk."""
    def __init__(self, start_note=60):
        self.last_note = start_note

    def generate_notes(self, measure, chord):
        notes = []
        for i in range(4): # Four quarter notes per measure
            # Higher probability of picking a note from the current chord
            if random.random() < 0.7:
                next_note = random.choice(chords[chord])
            else:
                next_note = random.choice(scale_notes)
            
            # Simple Markov: tend to move towards previous note
            if abs(next_note - self.last_note) > 7:
                next_note = self.last_note + random.choice([-2, 0, 2])
            
            self.last_note = next_note
            notes.append(mido.Message('note_on', note=next_note, velocity=70, time=0 if i == 0 else TICKS_PER_BEAT, channel=2))
            notes.append(mido.Message('note_off', note=next_note, velocity=0, time=0, channel=2))
        return notes

### 4. The Orchestrator Agent
This agent directs the band, telling them what chords to play and when. It collects all the generated notes.

In [ ]:
class OrchestratorAgent:
    def __init__(self, musicians):
        self.musicians = musicians
        self.song = []
        print(f"Orchestrator is ready with {len(musicians)} musicians.")

    def compose_song(self, num_measures, chord_progression):
        print("Composition started...")
        for measure in range(num_measures):
            current_chord = chord_progression[measure % len(chord_progression)]
            print(f"Measure {measure + 1}: Chord is {current_chord}")
            measure_notes = {'drums': [], 'bass': [], 'melody': []}
            for name, agent in self.musicians.items():
                measure_notes[name] = agent.generate_notes(measure, current_chord)
            self.song.append(measure_notes)
        print("Composition finished.")
        return self.song

### 5. Generate and Save the MIDI file

We'll now initialize all our agents, compose a song, and use the `mido` library to write the final output to `ai_composition.mid`.

In [ ]:
# 1. Create the musician agents
drummer = DrummerAgent()
bassist = BassistAgent()
melody_maker = MelodyAgent(start_note=note_map[KEY])

band = {
    'drums': drummer,
    'bass': bassist,
    'melody': melody_maker
}

# 2. Create the orchestrator and compose the song
orchestrator = OrchestratorAgent(band)
song_data = orchestrator.compose_song(16, CHORD_PROGRESSION)

# 3. Write to MIDI file
mid = mido.MidiFile(ticks_per_beat=TICKS_PER_BEAT)
mid.add_track('Drums')
mid.add_track('Bass')
mid.add_track('Melody')

for measure in song_data:
    for msg in measure['drums']:
        mid.tracks[0].append(msg)
    for msg in measure['bass']:
        mid.tracks[1].append(msg)
    for msg in measure['melody']:
        mid.tracks[2].append(msg)

output_filename = 'ai_composition.mid'
mid.save(output_filename)

print(f"\nSuccessfully saved composition to '{output_filename}'.")
print("You can now play this file in a music player or DAW (e.g., GarageBand, FL Studio) or an online sequencer.")